In [1]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:128,expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import math
import time
import json
import random
import csv
import gc
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Tuple, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch import amp

from h_lcm1_Copy1 import HyperbolicLCM


# ============================================================
# CONFIG
# ============================================================

@dataclass
class SFTConfig:
    datasets_to_run: Tuple[str, ...] = ("boolq_local",)

    hf_train_file: str = "/home/user/twovolume/Nisha/Finetune/boolq/train-00000-of-00001.parquet"
    hf_validation_file: str = "/home/user/twovolume/Nisha/Finetune/boolq/validation-00000-of-00001.parquet"

    out_dir: str = "runs/hlcm_boolq_sft"
    cache_dir: str = "boolq_local_cached_features_sft"

    ckpt_path: str = "runs/hyperbolic_cluster/checkpoints/ckpt_best.pt"
    normalizer_path: str = "normalizer.pt"

    encoder_name: str = "microsoft/deberta-v3-small"
    chunk_tok_len: int = 256
    seq_len: int = 12
    encoder_batch_size: int = 16
    conceptizer_device: str = "cpu"

    in_dim: int = 768
    model_dim: int = 4096
    num_heads: int = 32
    num_layers: int = 12
    ffn_mult: int = 4
    dropout: float = 0.30
    manifold_c: float = 0.002
    causal: bool = True
    input_scale: float = 0.05
    input_max_norm: float = 1.0

    finetune_mode: str = "last_blocks"   # "last_blocks" or "full"
    n_last_blocks: int = 2

    train_batch_size: int = 2
    eval_batch_size: int = 4
    grad_accum_steps: int = 8
    num_workers: int = 0

    sft_epochs: int = 5
    sft_lr: float = 3e-5
    sft_warmup_ratio: float = 0.06

    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    use_bf16: bool = True

    mcq_logit_temperature: float = 0.2
    label_smoothing: float = 0.02
    use_class_weights: bool = True

    ece_bins: int = 15

    min_valid_choices: int = 2
    seed: int = 42
    prefer_gpu_index: int = 0


# ============================================================
# BOOLQ TEMPLATES
# ============================================================

BOOLQ_YES_TEMPLATES = [
    "Answer: yes",
    "The correct answer is yes.",
    "Based on the passage, the answer is yes.",
    "The statement is supported by the passage.",
]

BOOLQ_NO_TEMPLATES = [
    "Answer: no",
    "The correct answer is no.",
    "Based on the passage, the answer is no.",
    "The statement is not supported by the passage.",
]


# ============================================================
# UTILS
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def pick_device(prefer_gpu_index: int = 0) -> torch.device:
    if not torch.cuda.is_available():
        return torch.device("cpu")
    n = torch.cuda.device_count()
    idx = 0 if n == 1 else max(0, min(prefer_gpu_index, n - 1))
    torch.cuda.set_device(idx)
    return torch.device(f"cuda:{idx}")


def build_amp_dtype(device: torch.device) -> torch.dtype:
    if device.type != "cuda":
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def save_checkpoint(payload: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(payload, path)


def append_dict_to_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    exists = os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def write_single_row_csv(path: str, row: Dict[str, Any]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        writer.writeheader()
        writer.writerow(row)


def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def gpu_mem_mb(device: Optional[torch.device] = None) -> Dict[str, float]:
    if not torch.cuda.is_available():
        return {"alloc_mb": 0.0, "reserved_mb": 0.0, "max_alloc_mb": 0.0}
    idx = device.index if device is not None and device.index is not None else torch.cuda.current_device()
    return {
        "alloc_mb": torch.cuda.memory_allocated(idx) / (1024 ** 2),
        "reserved_mb": torch.cuda.memory_reserved(idx) / (1024 ** 2),
        "max_alloc_mb": torch.cuda.max_memory_allocated(idx) / (1024 ** 2),
    }


def load_normalizer(normalizer_path: str, device: torch.device):
    if not normalizer_path or not os.path.exists(normalizer_path):
        return None, None
    obj = torch.load(normalizer_path, map_location="cpu")
    mu = obj["mu"].float().to(device)
    sigma = obj["sigma"].float().clamp_min(1e-4).to(device)
    return mu, sigma


def clone_state_dict_to_cpu(model: nn.Module) -> Dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


# ============================================================
# FREEZE / UNFREEZE
# ============================================================

def set_requires_grad(m: nn.Module, flag: bool):
    for p in m.parameters():
        p.requires_grad = flag


def freeze_all(model: nn.Module):
    set_requires_grad(model, False)


def unfreeze_all(model: nn.Module):
    set_requires_grad(model, True)


def unfreeze_last_blocks_hlcm(model: HyperbolicLCM, n_last: int):
    freeze_all(model)

    if not hasattr(model, "layers"):
        raise AttributeError("HyperbolicLCM must have attribute `layers`.")

    layers = list(model.layers)
    if len(layers) == 0:
        raise ValueError("HyperbolicLCM has no layers.")

    n_last = max(1, min(n_last, len(layers)))

    for blk in layers[-n_last:]:
        set_requires_grad(blk, True)

    for name, module in model.named_children():
        if name != "layers":
            set_requires_grad(module, True)


def apply_finetune_mode_hlcm(model: HyperbolicLCM, mode: str, n_last: int):
    if mode == "last_blocks":
        unfreeze_last_blocks_hlcm(model, n_last)
    elif mode == "full":
        unfreeze_all(model)
    else:
        raise ValueError(f"Unknown finetune_mode: {mode}")


# ============================================================
# CONCEPTIZER
# ============================================================

class DebertaConceptizer:
    def __init__(
        self,
        model_name: str,
        chunk_tok_len: int,
        seq_len: int,
        batch_size: int,
        device: torch.device,
    ):
        self.model_name = model_name
        self.chunk_tok_len = int(chunk_tok_len)
        self.seq_len = int(seq_len)
        self.batch_size = int(batch_size)
        self.device = device

        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.enc = AutoModel.from_pretrained(model_name).to(device)
        self.enc.eval()

        for p in self.enc.parameters():
            p.requires_grad = False

        self.hidden_size = int(self.enc.config.hidden_size)

    @torch.inference_mode()
    def _embed_chunk_texts(self, chunk_texts: List[str]) -> torch.Tensor:
        if len(chunk_texts) == 0:
            return torch.empty(0, self.hidden_size, dtype=torch.float32)

        inputs = self.tok(
            chunk_texts,
            padding=True,
            truncation=True,
            max_length=self.chunk_tok_len,
            return_tensors="pt",
        )
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        if self.device.type == "cuda":
            with amp.autocast(device_type="cuda", enabled=True, dtype=build_amp_dtype(self.device)):
                out = self.enc(**inputs).last_hidden_state[:, 0, :]
        else:
            out = self.enc(**inputs).last_hidden_state[:, 0, :]

        return out.detach().cpu().float()

    def _pack_into_chunk_texts(self, text: str) -> List[str]:
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        if not ids:
            return []

        chunks = []
        for i in range(0, len(ids), self.chunk_tok_len):
            chunks.append(
                self.tok.decode(
                    ids[i:i + self.chunk_tok_len],
                    clean_up_tokenization_spaces=True,
                )
            )
            if len(chunks) >= self.seq_len:
                break

        return chunks[:self.seq_len]

    @torch.inference_mode()
    def encode_text_fixed(self, text: str) -> Tuple[torch.Tensor, torch.Tensor]:
        chunk_texts = self._pack_into_chunk_texts(text)

        seq = torch.zeros(self.seq_len, self.hidden_size, dtype=torch.float32)
        pad = torch.ones(self.seq_len, dtype=torch.bool)

        if not chunk_texts:
            return seq, pad

        vecs = []
        for i in range(0, len(chunk_texts), self.batch_size):
            vecs.append(self._embed_chunk_texts(chunk_texts[i:i + self.batch_size]))

        vecs = torch.cat(vecs, dim=0)
        c = min(vecs.size(0), self.seq_len)

        seq[:c] = vecs[:c]
        pad[:c] = False

        return seq, pad


# ============================================================
# BOOLQ LOADING
# ============================================================

def load_boolq_local(cfg: SFTConfig):
    if not os.path.exists(cfg.hf_train_file):
        raise FileNotFoundError(f"Train parquet not found: {cfg.hf_train_file}")
    if not os.path.exists(cfg.hf_validation_file):
        raise FileNotFoundError(f"Validation parquet not found: {cfg.hf_validation_file}")

    raw = load_dataset(
        "parquet",
        data_files={
            "train": cfg.hf_train_file,
            "validation": cfg.hf_validation_file,
        },
    )

    return raw["train"], raw["validation"]


def normalize_boolq_example(ex: Dict[str, Any], min_valid_choices: int):
    passage = str(ex.get("passage", "")).strip()
    question = str(ex.get("question", "")).strip()
    answer = ex.get("answer", False)

    if question and not question.endswith("?"):
        question += "?"

    q_text = (
        "Task: Answer the yes/no question using the passage.\n"
        f"Passage: {passage}\n"
        f"Question: {question}"
    )

    choice_texts = ["no", "yes"]

    if len(choice_texts) < min_valid_choices:
        choice_texts = ["no", "yes"]

    label = 1 if bool(answer) else 0

    return q_text, choice_texts, label, passage, question


# ============================================================
# CACHE
# ============================================================

def cache_file_path(cfg: SFTConfig, dataset_name: str, split_name: str) -> str:
    safe_ds = dataset_name.replace("/", "_")
    ensure_dir(cfg.cache_dir)
    return os.path.join(
        cfg.cache_dir,
        f"{safe_ds}_{split_name}_tok{cfg.chunk_tok_len}_seq{cfg.seq_len}.pt",
    )


def build_or_load_cached_split(
    cfg: SFTConfig,
    dataset_name: str,
    split_name: str,
    hf_split,
    conceptizer: DebertaConceptizer,
) -> List[Dict[str, Any]]:
    path = cache_file_path(cfg, dataset_name, split_name)

    if os.path.exists(path):
        print(f"[cache] loading {path}")
        return torch.load(path, map_location="cpu")

    print(f"[cache] building {dataset_name} / {split_name}")

    rows = []
    skipped = 0

    for ex in tqdm(hf_split, desc=f"cache:{dataset_name}:{split_name}"):
        try:
            q_text, choice_texts, label, passage, question = normalize_boolq_example(
                ex,
                cfg.min_valid_choices,
            )

            q_seq, q_pad = conceptizer.encode_text_fixed(q_text)

            no_variants = [
                f"Passage: {passage}\nQuestion: {question}\n{t}"
                for t in BOOLQ_NO_TEMPLATES
            ]
            yes_variants = [
                f"Passage: {passage}\nQuestion: {question}\n{t}"
                for t in BOOLQ_YES_TEMPLATES
            ]

            all_choice_groups = []
            all_mask_groups = []

            for variants in [no_variants, yes_variants]:
                seqs, pads = [], []

                for txt in variants:
                    s, p = conceptizer.encode_text_fixed(txt)
                    seqs.append(s)
                    pads.append(p)

                all_choice_groups.append(torch.stack(seqs, dim=0))
                all_mask_groups.append(torch.stack(pads, dim=0))

            rows.append({
                "q": q_seq,
                "qmask": q_pad,
                "choices": torch.stack(all_choice_groups, dim=0),     # [2, V, T, D]
                "cmask": torch.stack(all_mask_groups, dim=0),         # [2, V, T]
                "choice_mask": torch.tensor([True, True], dtype=torch.bool),
                "label": int(label),
                "num_choices": 2,
            })

        except Exception as e:
            skipped += 1
            print(f"[warn] skipped example: {type(e).__name__}: {e}")

    torch.save(rows, path)
    print(f"[cache] saved {path} with {len(rows)} examples, skipped={skipped}")

    return rows


# ============================================================
# DATASET
# ============================================================

class CachedMCQDataset(Dataset):
    def __init__(self, rows: List[Dict[str, Any]]):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        r = self.rows[idx]
        return {
            "q": r["q"],
            "qmask": r["qmask"],
            "choices": r["choices"],
            "cmask": r["cmask"],
            "choice_mask": r["choice_mask"],
            "label": torch.tensor(r["label"], dtype=torch.long),
            "idx": idx,
        }


def cached_collate(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
    B = len(batch)
    T = batch[0]["q"].size(0)
    D = batch[0]["q"].size(1)
    K = batch[0]["choices"].size(0)
    V = batch[0]["choices"].size(1)

    q = torch.stack([x["q"] for x in batch], dim=0)
    qmask = torch.stack([x["qmask"] for x in batch], dim=0)

    choices = torch.zeros(B, K, V, T, D, dtype=torch.float32)
    cmask = torch.ones(B, K, V, T, dtype=torch.bool)
    choice_mask = torch.zeros(B, K, dtype=torch.bool)
    labels = torch.zeros(B, dtype=torch.long)
    idxs = torch.zeros(B, dtype=torch.long)

    for i, item in enumerate(batch):
        choices[i] = item["choices"]
        cmask[i] = item["cmask"]
        choice_mask[i] = item["choice_mask"]
        labels[i] = item["label"]
        idxs[i] = item["idx"]

    return {
        "q": q,
        "qmask": qmask,
        "choices": choices,
        "cmask": cmask,
        "choice_mask": choice_mask,
        "label": labels,
        "idx": idxs,
    }


# ============================================================
# MODEL
# ============================================================

def build_hlcm_from_cfg(cfg: SFTConfig) -> HyperbolicLCM:
    return HyperbolicLCM(
        in_dim=cfg.in_dim,
        model_dim=cfg.model_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        ffn_mult=cfg.ffn_mult,
        dropout=cfg.dropout,
        manifold_c=cfg.manifold_c,
        causal=cfg.causal,
        input_scale=cfg.input_scale,
        input_max_norm=cfg.input_max_norm,
    )


def load_pretrained_hlcm(cfg: SFTConfig, device: torch.device) -> HyperbolicLCM:
    model = build_hlcm_from_cfg(cfg).to(device)

    if not os.path.exists(cfg.ckpt_path):
        raise FileNotFoundError(f"Checkpoint not found: {cfg.ckpt_path}")

    obj = torch.load(cfg.ckpt_path, map_location="cpu")
    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj

    missing, unexpected = model.load_state_dict(state, strict=False)

    print(f"[load] loaded HLCM from {cfg.ckpt_path}")
    print(f"[load] missing keys: {len(missing)}")
    print(f"[load] unexpected keys: {len(unexpected)}")

    return model


# ============================================================
# LOGITS / LOSS
# ============================================================

def hlcm_last_tangent(
    model: HyperbolicLCM,
    x: torch.Tensor,
    pad_mask: torch.Tensor,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
) -> torch.Tensor:
    device = next(model.parameters()).device

    x = x.to(device, non_blocking=True)
    pad_mask = pad_mask.to(device, non_blocking=True)

    if mu is not None and sigma is not None:
        x = (x - mu.view(1, 1, -1)) / sigma.view(1, 1, -1)

    h = model(x)
    h_tan = model.manifold.logmap0(h)

    B, T, D = h_tan.shape
    out = torch.empty((B, D), device=h_tan.device, dtype=h_tan.dtype)

    for i in range(B):
        valid = (~pad_mask[i]).nonzero(as_tuple=False).view(-1)
        j = int(valid[-1].item()) if valid.numel() else 0
        out[i] = h_tan[i, j]

    return out


def mcq_logits_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    logit_temperature: float,
) -> torch.Tensor:
    device = next(model.parameters()).device

    q = batch["q"].to(device, non_blocking=True)
    qmask = batch["qmask"].to(device, non_blocking=True)
    choices = batch["choices"].to(device, non_blocking=True)
    cmask = batch["cmask"].to(device, non_blocking=True)
    choice_mask = batch["choice_mask"].to(device, non_blocking=True)

    B, K, V, T, D = choices.shape

    e_q = hlcm_last_tangent(model, q, qmask, mu, sigma)
    e_q = F.normalize(e_q, dim=-1)

    logits_per_choice = []

    for k in range(K):
        ch = choices[:, k]
        ch_mask = cmask[:, k]

        flat = ch.reshape(B * V, T, D)
        flat_mask = ch_mask.reshape(B * V, T)

        e_c = hlcm_last_tangent(model, flat, flat_mask, mu, sigma)
        e_c = e_c.reshape(B, V, -1)
        e_c = F.normalize(e_c, dim=-1)

        sims = torch.einsum("bd,bvd->bv", e_q, e_c)
        choice_logit = sims.mean(dim=1)

        logits_per_choice.append(choice_logit)

    logits = torch.stack(logits_per_choice, dim=1)
    logits = logits / max(logit_temperature, 1e-6)

    min_val = torch.finfo(logits.dtype).min
    logits = logits.masked_fill(~choice_mask, min_val)

    return logits


def compute_class_weights(rows: List[Dict[str, Any]]) -> torch.Tensor:
    counts = torch.zeros(2, dtype=torch.float32)

    for r in rows:
        counts[int(r["label"])] += 1.0

    counts = counts.clamp_min(1.0)
    weights = counts.sum() / (2.0 * counts)

    return weights


def mcq_loss_acc_hlcm(
    model: HyperbolicLCM,
    batch: Dict[str, torch.Tensor],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: SFTConfig,
    class_weights: Optional[torch.Tensor] = None,
):
    logits = mcq_logits_hlcm(
        model=model,
        batch=batch,
        mu=mu,
        sigma=sigma,
        logit_temperature=cfg.mcq_logit_temperature,
    )

    y = batch["label"].to(logits.device, non_blocking=True)
    weight = class_weights.to(logits.device) if class_weights is not None else None

    loss = F.cross_entropy(
        logits,
        y,
        weight=weight,
        label_smoothing=cfg.label_smoothing,
    )

    acc = (logits.argmax(dim=1) == y).float().mean()

    return loss, acc, logits


# ============================================================
# METRICS
# ============================================================

def compute_binary_metrics_from_probs(
    y_true: torch.Tensor,
    probs: torch.Tensor,
    ece_bins: int = 15,
) -> Dict[str, float]:
    """
    y_true: [N], labels 0 or 1
    probs:  [N, 2], softmax probabilities for [no, yes]
    """
    y_true = y_true.long().cpu()
    probs = probs.float().cpu()

    pred = probs.argmax(dim=1)
    conf = probs.max(dim=1).values

    tp = int(((pred == 1) & (y_true == 1)).sum().item())
    fp = int(((pred == 1) & (y_true == 0)).sum().item())
    fn = int(((pred == 0) & (y_true == 1)).sum().item())
    tn = int(((pred == 0) & (y_true == 0)).sum().item())

    accuracy = float((pred == y_true).float().mean().item())

    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)

    ranks = []
    for i in range(y_true.numel()):
        order = torch.argsort(probs[i], descending=True)
        rank = (order == y_true[i]).nonzero(as_tuple=False).view(-1)[0].item() + 1
        ranks.append(rank)

    mrr = sum(1.0 / r for r in ranks) / max(1, len(ranks))

    one_hot = F.one_hot(y_true, num_classes=2).float()
    brier = torch.mean(torch.sum((probs - one_hot) ** 2, dim=1)).item()

    ece = 0.0
    n = y_true.numel()
    correctness = (pred == y_true).float()

    bin_edges = torch.linspace(0.0, 1.0, steps=ece_bins + 1)

    for b in range(ece_bins):
        lo = bin_edges[b]
        hi = bin_edges[b + 1]

        if b == 0:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf > lo) & (conf <= hi)

        count = int(mask.sum().item())
        if count == 0:
            continue

        bin_acc = correctness[mask].mean().item()
        bin_conf = conf[mask].mean().item()
        ece += (count / max(1, n)) * abs(bin_acc - bin_conf)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "mrr": mrr,
        "brier": brier,
        "ece": ece,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }


@torch.no_grad()
def evaluate_hlcm(
    model: HyperbolicLCM,
    loader: Optional[DataLoader],
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: SFTConfig,
) -> Dict[str, float]:
    if loader is None:
        return {
            "loss": 0.0,
            "accuracy": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "mrr": 0.0,
            "brier": 0.0,
            "ece": 0.0,
        }

    model.eval()

    total_loss = 0.0
    total_n = 0

    all_probs = []
    all_labels = []

    for batch in loader:
        loss, acc, logits = mcq_loss_acc_hlcm(
            model=model,
            batch=batch,
            mu=mu,
            sigma=sigma,
            cfg=cfg,
            class_weights=None,
        )

        probs = F.softmax(logits.float(), dim=-1).detach().cpu()
        labels = batch["label"].detach().cpu()

        bs = labels.size(0)
        total_loss += float(loss.item()) * bs
        total_n += bs

        all_probs.append(probs)
        all_labels.append(labels)

    all_probs = torch.cat(all_probs, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    metrics = compute_binary_metrics_from_probs(
        y_true=all_labels,
        probs=all_probs,
        ece_bins=cfg.ece_bins,
    )

    metrics["loss"] = total_loss / max(1, total_n)

    return metrics


# ============================================================
# OPTIMIZER
# ============================================================

def make_optimizer_and_scheduler(
    trainable_params,
    lr: float,
    total_steps: int,
    warmup_ratio: float,
    weight_decay: float,
):
    trainable_params = list(trainable_params)

    if len(trainable_params) == 0:
        raise ValueError("No trainable parameters found.")

    opt = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay,
    )

    warmup_steps = int(warmup_ratio * total_steps)

    def lr_lambda(step: int):
        if step < warmup_steps:
            return step / max(1, warmup_steps)

        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        progress = min(1.0, progress)

        return 0.5 * (1.0 + math.cos(math.pi * progress))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    return opt, sched


# ============================================================
# SFT TRAINING
# ============================================================

def run_sft_hlcm(
    model: HyperbolicLCM,
    train_loader: DataLoader,
    eval_loader: DataLoader,
    out_dir: str,
    metadata: dict,
    mu: Optional[torch.Tensor],
    sigma: Optional[torch.Tensor],
    cfg: SFTConfig,
    device: torch.device,
    class_weights: Optional[torch.Tensor] = None,
):
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    steps_per_epoch = math.ceil(len(train_loader) / max(1, cfg.grad_accum_steps))
    total_steps = max(1, cfg.sft_epochs * steps_per_epoch)

    opt, sched = make_optimizer_and_scheduler(
        trainable_params=trainable_params,
        lr=cfg.sft_lr,
        total_steps=total_steps,
        warmup_ratio=cfg.sft_warmup_ratio,
        weight_decay=cfg.weight_decay,
    )

    use_amp = cfg.use_bf16 and device.type == "cuda"

    scaler = amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda" and use_amp and not torch.cuda.is_bf16_supported()),
    )

    train_csv = os.path.join(out_dir, "sft_train_log.csv")
    eval_csv = os.path.join(out_dir, "sft_eval_log.csv")

    base = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)

    print(
        f"[BASE] loss={base['loss']:.4f} "
        f"acc={base['accuracy']:.4f} "
        f"precision={base['precision']:.4f} "
        f"recall={base['recall']:.4f} "
        f"mrr={base['mrr']:.4f} "
        f"brier={base['brier']:.4f} "
        f"ece={base['ece']:.4f}"
    )

    append_dict_to_csv(eval_csv, {
        "phase": "base",
        "epoch": 0,
        "train_loss": "",
        "train_accuracy": "",
        **base,
    })

    best_metric = base["accuracy"]
    best_state = clone_state_dict_to_cpu(model)
    global_opt_step = 0
    start_time = time.time()

    for epoch in range(1, cfg.sft_epochs + 1):
        model.train()
        opt.zero_grad(set_to_none=True)

        epoch_loss_sum = 0.0
        epoch_acc_sum = 0.0
        epoch_count = 0

        pbar = tqdm(
            train_loader,
            desc=f"SFT epoch {epoch}/{cfg.sft_epochs}",
            dynamic_ncols=True,
        )

        for batch_idx, batch in enumerate(pbar, start=1):
            if device.type == "cuda":
                with amp.autocast(
                    device_type="cuda",
                    enabled=use_amp,
                    dtype=build_amp_dtype(device),
                ):
                    loss, acc, _ = mcq_loss_acc_hlcm(
                        model=model,
                        batch=batch,
                        mu=mu,
                        sigma=sigma,
                        cfg=cfg,
                        class_weights=class_weights,
                    )
                    loss_for_backward = loss / cfg.grad_accum_steps
            else:
                loss, acc, _ = mcq_loss_acc_hlcm(
                    model=model,
                    batch=batch,
                    mu=mu,
                    sigma=sigma,
                    cfg=cfg,
                    class_weights=class_weights,
                )
                loss_for_backward = loss / cfg.grad_accum_steps

            bs = batch["label"].size(0)
            epoch_loss_sum += float(loss.item()) * bs
            epoch_acc_sum += float(acc.item()) * bs
            epoch_count += bs

            if scaler.is_enabled():
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

            do_step = (
                batch_idx % cfg.grad_accum_steps == 0
                or batch_idx == len(train_loader)
            )

            if do_step:
                if scaler.is_enabled():
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    scaler.step(opt)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
                    opt.step()

                sched.step()
                opt.zero_grad(set_to_none=True)
                global_opt_step += 1

            pbar.set_postfix(
                loss=f"{epoch_loss_sum / max(1, epoch_count):.4f}",
                acc=f"{epoch_acc_sum / max(1, epoch_count):.4f}",
                lr=f"{opt.param_groups[0]['lr']:.2e}",
            )

        train_loss = epoch_loss_sum / max(1, epoch_count)
        train_accuracy = epoch_acc_sum / max(1, epoch_count)

        ev = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)
        cuda_cleanup()

        append_dict_to_csv(train_csv, {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "lr": opt.param_groups[0]["lr"],
            "global_opt_step": global_opt_step,
        })

        append_dict_to_csv(eval_csv, {
            "phase": "eval",
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            **ev,
        })

        print(
            f"[SFT][epoch {epoch}/{cfg.sft_epochs}] "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_accuracy:.4f} "
            f"eval_loss={ev['loss']:.4f} "
            f"eval_acc={ev['accuracy']:.4f} "
            f"precision={ev['precision']:.4f} "
            f"recall={ev['recall']:.4f} "
            f"mrr={ev['mrr']:.4f} "
            f"brier={ev['brier']:.4f} "
            f"ece={ev['ece']:.4f}"
        )

        if ev["accuracy"] > best_metric:
            best_metric = ev["accuracy"]
            best_state = clone_state_dict_to_cpu(model)

            save_checkpoint(
                {
                    "model": best_state,
                    "stage": "sft_best",
                    "epoch": epoch,
                    "global_opt_step": global_opt_step,
                    "best_eval_accuracy": best_metric,
                    "metrics": ev,
                    **metadata,
                },
                os.path.join(out_dir, "sft_best.pt"),
            )

            print("  saved sft_best.pt")

        save_checkpoint(
            {
                "model": clone_state_dict_to_cpu(model),
                "stage": "sft_last",
                "epoch": epoch,
                "global_opt_step": global_opt_step,
                "metrics": ev,
                **metadata,
            },
            os.path.join(out_dir, "sft_last.pt"),
        )

    total_minutes = (time.time() - start_time) / 60.0

    model.load_state_dict(best_state, strict=True)

    final_best_eval = evaluate_hlcm(model, eval_loader, mu, sigma, cfg)

    del opt
    del sched
    del scaler
    cuda_cleanup()

    return {
        "best_accuracy": best_metric,
        "best_state": best_state,
        "best_eval": final_best_eval,
        "total_minutes": total_minutes,
    }


# ============================================================
# TRAIN ONE DATASET
# ============================================================

def train_boolq_sft_hlcm(dataset_name: str, cfg: SFTConfig, device: torch.device):
    print(f"\n==================== {dataset_name} ====================")

    out_dir = os.path.join(cfg.out_dir, dataset_name.replace("/", "_"))
    ensure_dir(out_dir)
    ensure_dir(cfg.cache_dir)

    with open(os.path.join(out_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)

    conceptizer_device = torch.device(cfg.conceptizer_device)

    conceptizer = DebertaConceptizer(
        model_name=cfg.encoder_name,
        chunk_tok_len=cfg.chunk_tok_len,
        seq_len=cfg.seq_len,
        batch_size=cfg.encoder_batch_size,
        device=conceptizer_device,
    )

    train_hf, eval_hf = load_boolq_local(cfg)

    train_rows = build_or_load_cached_split(
        cfg=cfg,
        dataset_name=dataset_name,
        split_name="train",
        hf_split=train_hf,
        conceptizer=conceptizer,
    )

    eval_rows = build_or_load_cached_split(
        cfg=cfg,
        dataset_name=dataset_name,
        split_name="validation",
        hf_split=eval_hf,
        conceptizer=conceptizer,
    )

    del conceptizer
    cuda_cleanup()

    train_ds = CachedMCQDataset(train_rows)
    eval_ds = CachedMCQDataset(eval_rows)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.train_batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=cfg.eval_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(device.type == "cuda"),
        collate_fn=cached_collate,
        drop_last=False,
    )

    hlcm = load_pretrained_hlcm(cfg, device)
    apply_finetune_mode_hlcm(hlcm, cfg.finetune_mode, cfg.n_last_blocks)

    trainable_params = sum(p.numel() for p in hlcm.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in hlcm.parameters())

    print(f"[params] total={total_params:,} trainable={trainable_params:,}")

    if trainable_params == 0:
        raise ValueError("No trainable parameters found.")

    mu, sigma = load_normalizer(cfg.normalizer_path, device)

    class_weights = compute_class_weights(train_rows) if cfg.use_class_weights else None
    print("[class_weights]", None if class_weights is None else class_weights.tolist())

    metadata = {
        "dataset": dataset_name,
        "stage": "sft_only",
        "train_parquet": cfg.hf_train_file,
        "validation_parquet": cfg.hf_validation_file,
        "num_train_examples": len(train_ds),
        "num_eval_examples": len(eval_ds),
        "encoder_name": cfg.encoder_name,
        "chunk_tok_len": cfg.chunk_tok_len,
        "seq_len": cfg.seq_len,
        "ckpt_path": cfg.ckpt_path,
        "normalizer_path": cfg.normalizer_path,
        "finetune_mode": cfg.finetune_mode,
        "n_last_blocks": cfg.n_last_blocks,
        "sft_epochs": cfg.sft_epochs,
        "sft_lr": cfg.sft_lr,
        "sft_warmup_ratio": cfg.sft_warmup_ratio,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
        "label_smoothing": cfg.label_smoothing,
        "use_class_weights": cfg.use_class_weights,
        "ece_bins": cfg.ece_bins,
        "arch": {
            "in_dim": cfg.in_dim,
            "model_dim": cfg.model_dim,
            "num_heads": cfg.num_heads,
            "num_layers": cfg.num_layers,
            "ffn_mult": cfg.ffn_mult,
            "dropout": cfg.dropout,
            "manifold_c": cfg.manifold_c,
            "causal": cfg.causal,
            "input_scale": cfg.input_scale,
            "input_max_norm": cfg.input_max_norm,
        },
    }

    result = run_sft_hlcm(
        model=hlcm,
        train_loader=train_loader,
        eval_loader=eval_loader,
        out_dir=out_dir,
        metadata=metadata,
        mu=mu,
        sigma=sigma,
        cfg=cfg,
        device=device,
        class_weights=class_weights,
    )

    hlcm.load_state_dict(result["best_state"], strict=True)
    final_eval = evaluate_hlcm(hlcm, eval_loader, mu, sigma, cfg)
    mem = gpu_mem_mb(device)

    save_checkpoint(
        {
            "model": clone_state_dict_to_cpu(hlcm),
            "stage": "final_sft",
            "best_eval_accuracy": result["best_accuracy"],
            "final_eval": final_eval,
            "total_minutes": result["total_minutes"],
            "max_gpu_alloc_mb": mem["max_alloc_mb"],
            **metadata,
        },
        os.path.join(out_dir, "final_sft.pt"),
    )

    final_summary = {
        "dataset": dataset_name,
        "best_eval_accuracy": result["best_accuracy"],
        "final_eval_loss": final_eval["loss"],
        "final_eval_accuracy": final_eval["accuracy"],
        "final_eval_precision": final_eval["precision"],
        "final_eval_recall": final_eval["recall"],
        "final_eval_mrr": final_eval["mrr"],
        "final_eval_brier": final_eval["brier"],
        "final_eval_ece": final_eval["ece"],
        "tp": final_eval["tp"],
        "fp": final_eval["fp"],
        "fn": final_eval["fn"],
        "tn": final_eval["tn"],
        "total_minutes": result["total_minutes"],
        "max_gpu_alloc_mb": mem["max_alloc_mb"],
        "finetune_mode": cfg.finetune_mode,
        "n_last_blocks": cfg.n_last_blocks,
        "sft_epochs": cfg.sft_epochs,
        "sft_lr": cfg.sft_lr,
        "train_batch_size": cfg.train_batch_size,
        "eval_batch_size": cfg.eval_batch_size,
        "grad_accum_steps": cfg.grad_accum_steps,
        "mcq_logit_temperature": cfg.mcq_logit_temperature,
        "label_smoothing": cfg.label_smoothing,
        "use_class_weights": cfg.use_class_weights,
        "seq_len": cfg.seq_len,
        "chunk_tok_len": cfg.chunk_tok_len,
        "train_parquet": cfg.hf_train_file,
        "validation_parquet": cfg.hf_validation_file,
    }

    write_single_row_csv(
        os.path.join(out_dir, "final_summary.csv"),
        final_summary,
    )

    with open(os.path.join(out_dir, "final_summary.json"), "w", encoding="utf-8") as f:
        json.dump(final_summary, f, indent=2)

    print(
        f"\n[FINAL] dataset={dataset_name} "
        f"acc={final_eval['accuracy']:.4f} "
        f"precision={final_eval['precision']:.4f} "
        f"recall={final_eval['recall']:.4f} "
        f"mrr={final_eval['mrr']:.4f} "
        f"brier={final_eval['brier']:.4f} "
        f"ece={final_eval['ece']:.4f} "
        f"saved -> {out_dir}"
    )

    del hlcm
    del result
    cuda_cleanup()


# ============================================================
# MAIN
# ============================================================

def main():
    cfg = SFTConfig()

    ensure_dir(cfg.out_dir)
    ensure_dir(cfg.cache_dir)

    set_seed(cfg.seed)

    device = pick_device(cfg.prefer_gpu_index)

    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    print("Device:", device)
    print("Finetune mode:", cfg.finetune_mode)
    print("SFT epochs:", cfg.sft_epochs)
    print("Train batch size:", cfg.train_batch_size)
    print("Eval batch size:", cfg.eval_batch_size)
    print("Grad accumulation:", cfg.grad_accum_steps)
    print("SFT lr:", cfg.sft_lr)
    print("Temperature:", cfg.mcq_logit_temperature)
    print("Label smoothing:", cfg.label_smoothing)
    print("Use class weights:", cfg.use_class_weights)

    all_t0 = time.time()

    for ds_name in cfg.datasets_to_run:
        train_boolq_sft_hlcm(ds_name, cfg, device)

    total_all = (time.time() - all_t0) / 60.0

    print("\nAll done.")
    print("Outputs in:", cfg.out_dir)
    print(f"Total wall time: {total_all:.2f} min")


if __name__ == "__main__":
    main()

Device: cuda:0
Finetune mode: last_blocks
SFT epochs: 5
Train batch size: 2
Eval batch size: 4
Grad accumulation: 8
SFT lr: 3e-05
Temperature: 0.2
Label smoothing: 0.02
Use class weights: True

==================== boolq_local ====================


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache] loading boolq_local_cached_features_sft/boolq_local_train_tok256_seq12.pt
[cache] loading boolq_local_cached_features_sft/boolq_local_validation_tok256_seq12.pt
[load] loaded HLCM from runs/hyperbolic_cluster/checkpoints/ckpt_best.pt
[load] missing keys: 0
[load] unexpected keys: 0
[params] total=2,419,707,905 trainable=405,909,505
[class_weights] [1.3266253471374512, 0.8024344444274902]
[BASE] loss=0.6932 acc=0.4963 precision=0.6263 recall=0.4707 mrr=0.7482 brier=0.5000 ece=0.0054


SFT epoch 1/5: 100%|████| 4714/4714 [1:07:50<00:00,  1.16it/s, acc=0.4883, loss=0.6944, lr=2.84e-05]


[SFT][epoch 1/5] train_loss=0.6944 train_acc=0.4883 eval_loss=0.6903 eval_acc=0.6095 precision=0.6198 recall=0.9616 mrr=0.8047 brier=0.4971 ece=0.1038
  saved sft_best.pt


SFT epoch 2/5:  14%|▉      | 660/4714 [09:29<58:20,  1.16it/s, acc=0.5205, loss=0.6933, lr=2.77e-05]


KeyboardInterrupt: 